## Evaluate models finetuned for classification

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

from dual_ifm.tsimcne import metrics
from dual_ifm.utils import datasets, plot
from dual_ifm.classification.eval_finetune import get_all_metrics, load_embeddings, load_eval

# Run in parent dir
cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    os.chdir(cwd.parent)

pd.options.display.float_format = '{:,.3f}'.format

In [ ]:
prefix = 'tsimcneb'
dataset_dir = './datasets'
dataset_name = 'aptos'
feature_name = 'dr'
kfold = 1
img_size = (256, 256)  # Size of the input to the model
sample_size = None
sweep_name = f'clf_{prefix}_{dataset_name}_{feature_name}_{img_size[0]}'
experiment_name = f'dataset.kfold={kfold}_dataset={dataset_name}'

project_dir = Path.cwd()
checkpoints_dir = project_dir.joinpath('checkpoints')

### Load results and dataset

In [ ]:
preds, probs, targets, stats, wandb_id = load_eval(checkpoints_dir.joinpath(sweep_name), experiment_name)
X, X_2d, y = load_embeddings(checkpoints_dir.joinpath(sweep_name), experiment_name)

dataset, mapping = datasets.load_dataset(
    dataset_dir=dataset_dir,
    dataset_name=dataset_name,
    transform=None,
    image_size=img_size,
    feature_name=feature_name,
    drop_nan=False,
    sample_size=sample_size,
    split='all_holdout',
)

### Loss curve

In [ ]:
plot_labels = ['Loss', 'AUROC']

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for i in range(2):
    ax[i].plot(stats[f'{plot_labels[i].lower()}_train'], '-o', markersize=3)
    ax[i].plot(stats[f'{plot_labels[i].lower()}_val'], '-o', markersize=3)
    ax[i].legend(['Train', 'Val'])
    ax[i].set_title(plot_labels[i])
    ax[i].set_xlabel('Epoch')
    ax[i].set_ylabel(f'Average {plot_labels[i]}')

plt.tight_layout()
# fig.savefig(plot_file)

### Evaluate classification task

In [ ]:
auroc, auprc, acc, kappa = get_all_metrics(preds, probs, targets)

print(f'AUROC = {auroc:.2f}')
print(f'AUROC = {auprc:.2f}')
print(f'Blanced Accuracy = {acc:.2f}')
print(f'Kappa = {kappa:.2f}')

fig, ax = plt.subplots(1, 1, figsize=(7, 5))
cf_matrix = confusion_matrix(targets, preds)
mapping_cf = {k: v for k, v in mapping.items() if v != 'Missing'}
sns.heatmap(
    cf_matrix,
    xticklabels=list(mapping_cf.values()),
    yticklabels=list(mapping_cf.values()),
    annot=True,
    fmt='.2f',
    cmap='rocket_r',
    annot_kws={'size': 14},
    ax=ax,
)

ax.set_ylabel(f'Real {feature_name}')
ax.set_xlabel(f'Predicted {feature_name}')
ax.set_title('Confusion Matrix Test Set')
plt.tight_layout()

In [ ]:
# Select extra features to plot embeddings
if dataset_name == 'eyepacs':
    feature_names = ['dr', 'age', 'gender', 'ethnicity', 'field', 'camera']
elif dataset_name == 'areds':
    feature_names = ['age', 'gender', 'hbp', 'amd', 'diabetes', 'smoking']
elif dataset_name in ['idrid', 'aptos', 'deepdrid', 'messidor']:
    feature_names = ['dr']
elif dataset_name in ['papila', 'glaucoma']:
    feature_names = ['glaucoma']
elif dataset_name in ['fives']:
    feature_names = ['disease']
else:
    feature_names = ['Classes']

# Get extra features
y, mappings = dataset.get_feature(dataset.image_paths, feature_names, drop_nan=False)

if y.ndim == 1:
    y = np.expand_dims(y, axis=-1)
    mappings = [mappings]

name2label = {
    'dr': 'diabetic retinopathy',
    'dme': 'diabetic macular edema',
    'hbp': 'high blod pressure',
}
feature_label = [name2label.get(feature, feature) for feature in feature_names]

### Evaluate with KNN

In [ ]:
# Evaluate
X_train, X_test, y_train, y_test = train_test_split(X_2d, y)

print('KNN accuracy / R2')
for i in range(y.shape[1]):
    # Drop nans
    X_train_, y_train_, _ = plot.drop_nans(X_train, y_train[:, i])
    X_test_, y_test_, _ = plot.drop_nans(X_test, y_test[:, i])

    if feature_names[i] != 'age':
        metric = 100 * metrics.knn_acc(X_train_, X_test_, y_train_, y_test_)
        print(f'{feature_names[i].capitalize()} = {metric:.2f}%')
    else:
        metric = metrics.knn_reg(X_train_, X_test_, y_train_, y_test_)
        print(f'{feature_names[i].capitalize()} = {metric:.3f}')

### Visualize 2D embeddings

In [ ]:
if dataset_name in ['eyepacs', 'areds']:
    n_subplots = (2, 3)
    if dataset_name == 'eyepacs':
        imbalanced = [True, False, True, True, True, False]
        categorical = [True, False, True, True, True, True]
    elif dataset_name == 'areds':
        imbalanced = [False, False, True, False, True, False]
        categorical = [False, True, True, False, True, True]
else:
    n_subplots = (1, 1)
    imbalanced = [False]
    categorical = [False]

fig_width = 'full'
fig_height_ratio = 0.8

plot.plot_embeddings(
    X_2d,
    y,
    mappings,
    None,
    n_subplots,
    fig_width,
    fig_height_ratio,
    feature_names,
    imbalanced,
    categorical,
    s_marker=40
)

### Compute K-fold results

In [ ]:
backbone_names = ['BagNet33']
weight_names = ['ImageNet', 'SimCLR', 't-SimCNE', 't-SimCNEx']  #
dataset_names = ['EyePACS', 'AREDS', 'UKB'] # In pretraining
# dataset_names = ['APTOS', 'DeepDRiD', 'IDRiD', 'Messidor', 'Glaucoma', 'PAPILA', 'FIVES']

df_rows = []
for dataset_name in dataset_names:
    if dataset_name in ['AREDS']:
        feature_name = 'amd'
    elif dataset_name in ['Glaucoma', 'PAPILA']:
        feature_name = 'glaucoma'
    elif dataset_name in ['FIVES']:
        feature_name = 'disease'
    else:
        feature_name = 'dr'

    for backbone_name in backbone_names:
        for weight_name in weight_names:
            # Experiment name
            prefix = f'{weight_name.replace("-", "").lower()}{backbone_name[0].lower()}'
            sweep_name = f'clf_{prefix}_{dataset_name.lower()}_{feature_name}_256'           

            for fold in range(1, 6):
                experiment_name = f'dataset.kfold={fold}_dataset={dataset_name.lower()}'
                if os.path.exists(checkpoints_dir.joinpath(sweep_name, experiment_name + '.json')):
                    preds, probs, targets, _, _ = load_eval(checkpoints_dir.joinpath(sweep_name), experiment_name)
                    auroc, auprc, acc, kappa = get_all_metrics(preds, probs, targets)
                    row = {
                        'Model': f'{backbone_name} {weight_name}',
                        'Dataset': dataset_name,
                        'kfold': fold,
                        'AUROC': auroc,
                        'AUPRC': auprc,
                        'Balanced Accuracy': acc,
                        'Kappa': kappa,
                    }
                    df_rows.append(row)

df = pd.DataFrame(df_rows)

In [ ]:
df_groupby = df.drop(columns='kfold').groupby(by=['Dataset', 'Model']).agg(('mean', 'std'))
df_groupby

In [ ]:
# Display one metric + sd
df_metric = df_groupby['AUROC'].copy()
df_metric['mean_sd'] = df_metric['mean'].round(3).astype(str) + ' ± ' + df_metric['std'].round(3).astype(str)
table_metric = df_metric['mean_sd'].reset_index().pivot(index='Dataset', columns='Model', values='mean_sd')
table_metric = table_metric.reindex(dataset_names)
table_metric

In [ ]:
# Best fold
df.loc[df.groupby(['Dataset', 'Model'])['AUROC'].idxmax()]

In [ ]:
# Per dataset
df[df['Dataset'] == 'PAPILA']

In [ ]:
print(table_metric.to_csv(sep='\t'))

In [ ]:
latex_table = table_metric.to_latex(
    float_format='%.3f',
    multicolumn=True,
    multirow=True,
    caption='caption',
    label='tab:label',
)

print(latex_table)